# Airbnb London Scraper — Full 33 Boroughs, Price-Band Split

**Fixes the pagination problem you hit** (duplicates from page 3 onward, ~1,000 rows total).

Root cause: a single search query per borough only returns ~2 pages of unique results before Airbnb starts re-serving the same cards — that's a per-query depth limit, not a shortage of London listings.

**Fix:** each borough is now searched once per **price band** (`price_min`/`price_max` — genuine Airbnb search parameters), so each band is a separate query with its own fresh ~2 pages. 33 boroughs × 6 price bands × ~2 pages should land you in the 5,000–6,000 range.

**Before the full run:**
1. Install dependencies: `pip install selenium beautifulsoup4 pandas openpyxl webdriver-manager`
2. Run the pagination depth test (last function in the cell below) on 2–3 boroughs to confirm 2 pages/query is still right — see the docstring on `test_pagination_depth()` for a ready-to-run example.
3. Then run the cell as-is (`if __name__ == "__main__"` block runs the full 33 × 6 sweep automatically).

**If you need to stop and resume:** just re-run the same cell — `checkpoint_listings.csv` and `checkpoint_progress.json` track what's already done and pick up where you left off, per (borough, price band) query.


In [1]:
"""
Airbnb Listings Scraper (Selenium + BeautifulSoup) — 33 boroughs, price-band split
------------------------------------------------------------------------------------
Scrapes publicly visible listing data from Airbnb search results pages
(title, price, rating, review count, room type, location snippet, listing URL)
and saves everything to an Excel file.

WHAT CHANGED FROM THE PREVIOUS VERSION AND WHY:
Running one search per borough tops out at ~2 pages before Airbnb starts
re-serving page 1/2 results (confirmed by your own test: ~1,000 rows across
33 boroughs, duplicates from page 3 onward). That's a real depth limit on a
SINGLE search query, not a shortage of London listings — InsideAirbnb's own
snapshot shows 80,000+ active listings citywide, so the supply is there;
one query per borough just wasn't reaching enough of it.

Fix: instead of one search per borough, each borough is now searched
SEVERAL TIMES, once per price band (price_min / price_max are real Airbnb
search URL parameters). Each price band is a distinct query to Airbnb, so
each one gets its own ~2 pages of unique results before hitting the same
depth limit — multiplying unique listings per borough instead of fighting
the limit. Splitting by price also has a side benefit for your dissertation:
it guards against low-price listings being pushed off the first couple of
pages by Airbnb's default relevance/quality ranking, so your final sample
isn't skewed toward whatever the algorithm ranks highest.

Math: 33 boroughs x 6 price bands x ~2 pages x ~18 unique cards/page
(before de-dup) is comfortably in the 5,000-6,000+ range this is tuned for.
Actual yield varies by borough (Inner London boroughs are denser than
outer ones) - the checkpoint file lets you see the running total live and
add more bands/pages if you're short, without re-scraping what you already
have.

BEFORE YOU RUN THIS:
1. Install Google Chrome on your machine (if not already installed).
2. Install the Python packages:
   pip install selenium beautifulsoup4 pandas openpyxl webdriver-manager
3. Run the PAGINATION DEPTH TEST cell first (bottom of this file) on 2-3
   boroughs to confirm 2 pages/query is still the right depth before
   committing to the full run - Airbnb's behaviour can change.

ETHICAL / LEGAL NOTE FOR YOUR PROJECT WRITE-UP:
- This script only reads publicly visible search-result HTML — no login,
  no bypassing paywalls or CAPTCHAs, no private data.
- It rate-limits requests (random delays) to avoid hammering Airbnb's
  servers, which is both good etiquette and reduces the chance of
  getting blocked.
- Airbnb's Terms of Service restrict automated scraping. For a university
  project this is normally fine under fair-use/academic-research
  justification, but you should say explicitly in your methodology
  section that: (a) you only collected publicly available fields relevant
  to your research questions, (b) you rate-limited your requests, and
  (c) no personal host/guest data was collected.
- If Airbnb changes their page layout, the CSS selectors below WILL need
  updating — that's normal for web scraping and worth noting as a
  limitation in your report.
- Document the price-band splitting approach explicitly in your methods
  section too: it's a deliberate sampling design choice made to overcome
  a platform-imposed pagination limit, not an attempt to bypass any
  access control.
"""

import time
import random
import re
import os
import json
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException, InvalidSessionIdException

# Restart Chrome after this many completed queries, even if nothing has
# gone wrong yet. Headless Chrome's memory footprint grows steadily over
# a long run (198 queries here); left unchecked it eventually gets killed
# by the OS partway through, which is what was cutting the run short.
RESTART_EVERY_N_QUERIES = 25


# ============================================================
# CONFIG — edit these to suit your project
# ============================================================
# Full list of all 33 London boroughs (official Inner/Outer classification).
FULL_LOCATIONS = [
    "Barking and Dagenham, London, United Kingdom", "Barnet, London, United Kingdom",
    "Bexley, London, United Kingdom", "Brent, London, United Kingdom",
    "Bromley, London, United Kingdom", "Camden, London, United Kingdom",
    "Croydon, London, United Kingdom", "Ealing, London, United Kingdom",
    "Enfield, London, United Kingdom", "Greenwich, London, United Kingdom",
    "Hackney, London, United Kingdom", "Hammersmith and Fulham, London, United Kingdom",
    "Haringey, London, United Kingdom", "Harrow, London, United Kingdom",
    "Havering, London, United Kingdom", "Hillingdon, London, United Kingdom",
    "Hounslow, London, United Kingdom", "Islington, London, United Kingdom",
    "Kensington and Chelsea, London, United Kingdom", "Kingston upon Thames, London, United Kingdom",
    "Lambeth, London, United Kingdom", "Lewisham, London, United Kingdom",
    "Merton, London, United Kingdom", "Newham, London, United Kingdom",
    "Redbridge, London, United Kingdom", "Richmond upon Thames, London, United Kingdom",
    "Southwark, London, United Kingdom", "Sutton, London, United Kingdom",
    "Tower Hamlets, London, United Kingdom", "Waltham Forest, London, United Kingdom",
    "Wandsworth, London, United Kingdom", "Westminster, London, United Kingdom",
    "City of London, United Kingdom",
]

LOCATIONS = FULL_LOCATIONS

# ------------------------------------------------------------------
# PRICE BANDS — the mechanism that actually solves the pagination
# problem. Each (min, max) pair becomes a SEPARATE Airbnb search query
# per borough (price_min / price_max URL params), so each band gets its
# own ~2 pages of unique results rather than all competing for the same
# top ~30-35 listings a single borough-wide search returns.
#
# Bands are wider at the low/high ends and narrower in the middle, where
# most London listings cluster (roughly £75-250/night), so no single
# band is overloaded with far more listings than PAGES_PER_QUERY can
# reach. Adjust freely - narrower bands = more queries = more listings,
# at the cost of more runtime.
# ------------------------------------------------------------------
PRICE_BANDS = [
    (None, 75),
    (75, 120),
    (120, 175),
    (175, 250),
    (250, 400),
    (400, None),
]

# Pages per (borough, price band) query. Kept at 2 based on your own
# diagnostic run (duplicates started on page 3). Re-confirm with
# test_pagination_depth() below before a full run if you change bands,
# dates, or notice Airbnb's behaviour has changed.
PAGES_PER_QUERY = 2

# IMPORTANT: fixed dates are required so every listing quotes the SAME
# number of nights. Without this, Airbnb assigns each card a random
# check-in/check-out window, and the total price becomes incomparable
# across rows (a 4-night £400 stay vs a 10-night £1000 stay).
CHECKIN = "2026-09-01"
CHECKOUT = "2026-09-05"               # 4 nights - change both together if you want a different length
EXPECTED_NIGHTS = (
    datetime.strptime(CHECKOUT, "%Y-%m-%d") - datetime.strptime(CHECKIN, "%Y-%m-%d")
).days
HEADLESS = True                       # False = watch the browser work (good for debugging)
OUTPUT_FILE = f"airbnb_london_full33_pricebands_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
MIN_DELAY, MAX_DELAY = 3, 7           # seconds between page loads (be polite)

# ------------------------------------------------------------------
# RESUME / CHECKPOINTING
# A 33-borough x 6-band run is long enough that a crash, a CAPTCHA
# block, or a dropped connection partway through would otherwise mean
# starting over. Every (borough, price band) query's listings are
# appended to CHECKPOINT_CSV as soon as that query finishes, and its
# key is recorded in PROGRESS_JSON. If scrape_airbnb() is re-run, it
# skips any (borough, band) combination already marked done and picks
# up where it left off, so nothing already scraped is lost or
# re-scraped.
# ------------------------------------------------------------------
CHECKPOINT_CSV = "checkpoint_listings.csv"
PROGRESS_JSON = "checkpoint_progress.json"

# Full official Inner/Outer London classification (33 boroughs incl. City
# of London), so zone_for_location() never returns None for any borough in
# FULL_LOCATIONS above.
BOROUGH_ZONE_MAP = {
    # Inner London
    "Camden": "Inner London",
    "Islington": "Inner London",
    "Hackney": "Inner London",
    "Westminster": "Inner London",
    "Tower Hamlets": "Inner London",
    "Southwark": "Inner London",
    "Lambeth": "Inner London",
    "Wandsworth": "Inner London",
    "Hammersmith and Fulham": "Inner London",
    "Kensington and Chelsea": "Inner London",
    "Lewisham": "Inner London",
    "Greenwich": "Inner London",
    "City of London": "Inner London",
    # Outer London
    "Croydon": "Outer London",
    "Bromley": "Outer London",
    "Barnet": "Outer London",
    "Ealing": "Outer London",
    "Enfield": "Outer London",
    "Havering": "Outer London",
    "Hillingdon": "Outer London",
    "Redbridge": "Outer London",
    "Sutton": "Outer London",
    "Bexley": "Outer London",
    "Barking and Dagenham": "Outer London",
    "Brent": "Outer London",
    "Harrow": "Outer London",
    "Hounslow": "Outer London",
    "Kingston upon Thames": "Outer London",
    "Merton": "Outer London",
    "Richmond upon Thames": "Outer London",
    "Waltham Forest": "Outer London",
    "Haringey": "Outer London",
    "Newham": "Outer London",
}


def build_search_url(location, checkin=None, checkout=None, price_min=None, price_max=None):
    """
    Builds an Airbnb search URL for the FIRST page of a given (location,
    price band) query. price_min / price_max are genuine Airbnb search
    parameters (the same ones the price filter slider in the UI sets) -
    this is what actually produces a DIFFERENT result set per band,
    rather than a URL trick that Airbnb's backend ignores.

    Pagination beyond page 1 is still done by clicking the actual "Next"
    button in the browser (see scrape_one_location), not by building a
    different URL - Airbnb's backend ignores an items_offset param.
    """
    base = "https://www.airbnb.co.uk/s/{}/homes".format(location.replace(", ", "--").replace(" ", "-"))
    params = []
    if checkin:
        params.append(f"checkin={checkin}")
    if checkout:
        params.append(f"checkout={checkout}")
    if price_min is not None:
        params.append(f"price_min={price_min}")
    if price_max is not None:
        params.append(f"price_max={price_max}")
    return base + ("?" + "&".join(params) if params else "")


def start_driver(headless=True):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver


def is_driver_alive(driver):
    """Cheap check that the Chrome session behind `driver` is still
    responding. A crashed/killed Chrome process leaves the Selenium
    session object in place but every call to it raises - this catches
    that before wasting a query on a dead browser."""
    try:
        _ = driver.current_url
        return True
    except Exception:
        return False


def restart_driver(driver, headless=True):
    """Cleanly tears down the current driver (if it's still there to
    tear down) and starts a fresh Chrome session."""
    try:
        driver.quit()
    except Exception:
        pass
    return start_driver(headless=headless)


def scroll_page(driver, pause=1.5, scrolls=6):
    """Airbnb lazy-loads cards as you scroll — scroll down in steps to force them in."""
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height


def parse_total_price(raw_text):
    """
    Extracts the numeric TOTAL stay price - i.e. what a guest actually pays.
    Airbnb shows a struck-through original price next to the discounted
    price for any listing with an active discount, e.g. "£870 £751 total".
    Anchor specifically to the number immediately preceding "total"; only
    fall back to "last number found" if no "total" marker is present.
    """
    if not raw_text:
        return None

    m = re.search(r"([\d,]+(?:\.\d+)?)\s*total", raw_text, re.IGNORECASE)
    if m:
        return float(m.group(1).replace(",", ""))

    numbers = re.findall(r"[\d,]+", raw_text.replace("£", ""))
    if numbers:
        return float(numbers[-1].replace(",", ""))
    return None


def parse_nights(full_text):
    """
    Looks for a 'for X night(s)' phrase in the card text. Returns None if
    not found, in which case we fall back to EXPECTED_NIGHTS from config.
    """
    m = re.search(r"for\s+(\d+)\s+nights?", full_text, re.IGNORECASE)
    return int(m.group(1)) if m else None


# Known hotel/aparthotel brands that show up in Airbnb search results
# alongside peer-to-peer host listings. Hedonic pricing models (Rosen,
# and the Airbnb-specific literature that follows it) are built around
# peer-to-peer listings - a commercial hotel product priced by different
# logic sitting in the same regression can distort the coefficients, so
# these need to be flagged/filterable rather than silently included.
HOTEL_BRAND_KEYWORDS = [
    "point a", "easyhotel", "staycity", "roomzzz", "premier inn",
    "travelodge", "holiday inn", "ibis", "hilton", "marriott",
    "novotel", "mercure", "hampton by hilton", "citizenm", "z hotel",
    "the z hotel", "aparthotel", "apart hotel", "premier suites",
    "cove aparthotels", "locke", "yotel",
]

# Standard category buckets, checked in this order (first match wins).
PROPERTY_TYPE_KEYWORDS = [
    ("Hotel", HOTEL_BRAND_KEYWORDS + ["hotel room", "room in hotel", "hotel"]),
    ("Room", ["private room", "shared room", "room in "]),
    ("Flat", ["rental unit", "apartment", "flat", "condo", "loft", "serviced apartment"]),
    ("House", ["home", "house", "townhouse", "cottage", "bungalow", "villa"]),
]


def classify_property_type(full_text, aria_label):
    """
    Classifies each listing into a standard bucket: Room, Flat, House,
    Hotel, or Other/Unknown. Searches the card's aria-label (accessibility
    text, which tends to carry Airbnb's actual type wording, e.g. "Entire
    rental unit in...") plus the full concatenated card text as a fallback
    - NOT the title, which is just the host's chosen nickname.

    Order matters: Hotel is checked first so a hotel brand name doesn't
    get miscategorised as "Room" or "Flat" just because its listing also
    contains those words.
    """
    haystack = f"{aria_label or ''} {full_text or ''}".lower()

    for category, keywords in PROPERTY_TYPE_KEYWORDS:
        if any(kw in haystack for kw in keywords):
            is_hotel = category == "Hotel"
            return category, is_hotel

    return "Other/Unknown", False


def parse_listing_card(card):
    """
    Pulls the fields we care about out of one listing card's HTML. Builds
    ONE concatenated text blob (`full_text`) from the entire card and runs
    regexes against that blob rather than a single BeautifulSoup text node,
    since Airbnb frequently splits rating and review count into separate
    text nodes in the DOM.
    """
    data = {}
    full_text = card.get_text(" ", strip=True)

    # Title / name
    title_tag = card.select_one('[data-testid="listing-card-title"]')
    data["title"] = title_tag.get_text(strip=True) if title_tag else None

    # Price - grab the raw text, then split into total price and nights
    price_tag = card.select_one('span[class*="price"], div[data-testid*="price"]')
    price_text = price_tag.get_text(" ", strip=True) if price_tag else full_text
    data["price_raw"] = price_text

    total_price = parse_total_price(price_text)
    nights = parse_nights(full_text) or parse_nights(price_text)
    data["nights"] = nights if nights else EXPECTED_NIGHTS
    data["nights_source"] = "parsed_from_card" if nights else "assumed_from_search_dates"
    data["price_gbp_total"] = total_price
    data["price_gbp_per_night"] = (
        round(total_price / data["nights"], 2) if total_price and data["nights"] else None
    )

    # Rating + review count - anchor to aria-label elements (Airbnb's
    # standard pattern is "4.85 out of 5 average rating, 32 reviews"),
    # and search rating/review count independently since Airbnb often
    # splits them into separate nodes.
    aria_texts = [el.get("aria-label") for el in card.find_all(attrs={"aria-label": True})]
    aria_texts = [t for t in aria_texts if t]

    data["rating"] = None
    data["review_count"] = None

    for src in aria_texts + [full_text]:
        m = re.search(r"(\d\.\d{1,2})\s*out of 5", src, re.IGNORECASE)
        if m and 1.0 <= float(m.group(1)) <= 5.0:
            data["rating"] = float(m.group(1))
            break

    for src in aria_texts + [full_text]:
        m = re.search(r"(\d+)\s*reviews?", src, re.IGNORECASE)
        if m:
            data["review_count"] = int(m.group(1))
            break
    if data["review_count"] is None:
        for src in aria_texts + [full_text]:
            m = re.search(r"\((\d+)\)", src)
            if m:
                data["review_count"] = int(m.group(1))
                break

    # Missing rating/reviews are usually genuinely new listings rather than
    # a parsing failure - flag them explicitly so this is a documented,
    # deliberate modelling choice (e.g. imputation) rather than a silent gap.
    data["is_new_listing"] = data["rating"] is None and data["review_count"] is None

    # Listing URL + aria-label
    link_tag = card.select_one("a[href*='/rooms/']")
    data["url"] = ("https://www.airbnb.co.uk" + link_tag["href"]) if link_tag else None
    aria_label = link_tag.get("aria-label") if link_tag else None

    # Clean numeric room ID, stripped of query string. Each duplicate card
    # carries a different tracking ID in its URL query string, so two rows
    # for the SAME listing look like different URLs under plain URL-based
    # dedup. room_id fixes this, and is also what lets us safely dedup
    # across price bands (the same listing can legitimately surface in two
    # adjacent bands near a boundary).
    room_id_match = re.search(r"/rooms/(\d+)", data["url"] or "")
    data["room_id"] = room_id_match.group(1) if room_id_match else None

    # Property type / hotel flag - standard bucket classification
    data["property_type"], data["is_hotel_brand"] = classify_property_type(full_text, aria_label)

    return data


def zone_for_location(location):
    """Maps a search location string to Inner/Outer London using BOROUGH_ZONE_MAP."""
    for borough, zone in BOROUGH_ZONE_MAP.items():
        if borough.lower() in location.lower():
            return zone
    return None


def go_to_next_page(driver, page, max_pages):
    """
    Clicks the 'Next' button and BLOCKS until the next page has actually
    finished rendering, instead of trusting a fixed sleep before the next
    loop iteration re-reads the page. Airbnb's pagination re-renders the
    results list client-side (the URL never changes - only the DOM node
    identity does), so a two-stage explicit wait is used:
      1. staleness_of the current first card - confirms the OLD cards
         were actually torn down by the click.
      2. presence_of a NEW listing-card-title - confirms the next page's
         cards have actually mounted.
    Returns True if the next page is ready to be read, False if pagination
    should stop for this query.
    """
    try:
        next_btn = driver.find_element(
            By.CSS_SELECTOR,
            'a[aria-label="Next"], button[aria-label="Next"], '
            'a[aria-label="Next page"], nav[aria-label*="pagination" i] a:last-child'
        )
    except Exception:
        print("    PAGINATION: 'Next' button not found in DOM - reached the last "
              "available page for this query.")
        return False

    try:
        old_first_card = driver.find_element(By.CSS_SELECTOR, '[data-testid="listing-card-title"]')
    except Exception:
        old_first_card = None

    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
    time.sleep(0.5)

    try:
        next_btn.click()
        print(f"  [Page {page + 2}/{max_pages}] PAGINATION: 'Next' button found and clicked.")
    except Exception as e:
        print(f"    PAGINATION: click on 'Next' button failed ({e}) - stopping this query.")
        return False

    if old_first_card is not None:
        try:
            WebDriverWait(driver, 15).until(EC.staleness_of(old_first_card))
            print("    PAGINATION: previous page's cards went stale - re-render confirmed.")
        except Exception:
            print("    PAGINATION WARNING: previous page's cards never went stale after 15s "
                  "- the click may not have registered. Proceeding to check for new cards anyway.")

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="listing-card-title"]'))
        )
        print("    PAGINATION: new listing cards detected on the next page.")
    except Exception:
        print("    PAGINATION: no new cards appeared after clicking 'Next' - stopping this query.")
        return False

    return True


def scrape_one_query(driver, location, price_min, price_max, max_pages, checkin=None, checkout=None):
    """
    Scrapes up to max_pages for ONE (location, price band) query. This is
    the unit of work that gets checkpointed - each call is one distinct
    Airbnb search, so it gets its own ~2 pages of unique results rather
    than competing with the rest of the borough's listings for the same
    top ~30-35 cards a single borough-wide search returns.
    """
    listings = []
    url = build_search_url(location, checkin, checkout, price_min, price_max)
    band_label = f"£{price_min if price_min is not None else 0}-{price_max if price_max is not None else '+'}"
    print(f"  [Page 1/{max_pages}] Loading: {url}")
    driver.get(url)

    seen_room_ids = set()

    for page in range(max_pages):
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="listing-card-title"]'))
            )
        except Exception:
            print("    No listing cards found on this page — stopping this query "
                  "(may be end of results for this band, or Airbnb blocked/changed layout).")
            break

        scroll_page(driver)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # card-container is NESTED INSIDE itemListElement on Airbnb's actual
        # markup - try the more specific inner selector first so listings
        # aren't picked up twice (once as outer wrapper, once as inner card).
        cards = soup.select('div[data-testid="card-container"]')
        if not cards:
            cards = soup.select('[itemprop="itemListElement"]')
        if not cards:
            cards = soup.select('div:has([data-testid="listing-card-title"])')

        print(f"    Found {len(cards)} listing cards")

        # Check the WHOLE page's room_ids against everything seen so far in
        # THIS query, not just the first card - Airbnb pins a couple of
        # sponsored listings at the top of every page, which can legitimately
        # repeat even when the rest of the page is genuinely new. Only treat
        # it as a real stall if NONE of this page's listings are new.
        page_room_ids = {
            m.group(1) for c in cards
            if (a := c.select_one("a[href*='/rooms/']"))
            for m in [re.search(r"/rooms/(\d+)", a["href"])] if m
        }
        new_ids = page_room_ids - seen_room_ids
        if page > 0 and page_room_ids and not new_ids:
            print("    WARNING: none of this page's listings are new - pagination "
                  "did not actually advance. Stopping this query.")
            break
        seen_room_ids |= page_room_ids

        for card in cards:
            listing = parse_listing_card(card)
            listing["search_location"] = location
            listing["zone"] = zone_for_location(location)
            listing["price_band_min"] = price_min
            listing["price_band_max"] = price_max
            listing["price_band_label"] = band_label
            listing["page_scraped"] = page + 1
            listing["scraped_at"] = datetime.now().isoformat(timespec="seconds")
            listings.append(listing)

        if page < max_pages - 1:
            advanced = go_to_next_page(driver, page, max_pages)
            if not advanced:
                break

        delay = random.uniform(MIN_DELAY, MAX_DELAY)
        print(f"    Waiting {delay:.1f}s...")
        time.sleep(delay)

    return listings


def query_key(location, price_min, price_max):
    """Unique resume key for a (location, price band) query."""
    return f"{location} || price_min={price_min} || price_max={price_max}"


def load_progress():
    """Returns the set of query keys already scraped and checkpointed."""
    if os.path.exists(PROGRESS_JSON):
        with open(PROGRESS_JSON, "r") as f:
            return set(json.load(f).get("completed_queries", []))
    return set()


def mark_query_done(key):
    """Appends one query key to the progress file, preserving prior entries."""
    completed = load_progress()
    completed.add(key)
    with open(PROGRESS_JSON, "w") as f:
        json.dump({"completed_queries": sorted(completed)}, f, indent=2)


def append_checkpoint(listings):
    """Appends this query's rows to CHECKPOINT_CSV, writing the header
    only if the file doesn't exist yet."""
    if not listings:
        return
    df = pd.DataFrame(listings)
    write_header = not os.path.exists(CHECKPOINT_CSV)
    df.to_csv(CHECKPOINT_CSV, mode="a", header=write_header, index=False)


def load_checkpoint():
    """Loads everything scraped so far from CHECKPOINT_CSV, or an empty
    DataFrame if no checkpoint exists yet (first run)."""
    if os.path.exists(CHECKPOINT_CSV):
        return pd.read_csv(CHECKPOINT_CSV)
    return pd.DataFrame()


def scrape_airbnb(locations, price_bands, pages_per_query, checkin=None, checkout=None, headless=True):
    """
    Resumable version: iterates every (borough, price band) combination,
    skipping any already scraped in a previous run via PROGRESS_JSON. Each
    query's listings are written to CHECKPOINT_CSV the moment that query
    finishes, not held in memory until the very end, so a failure partway
    through never loses already-scraped data.

    A failure on ONE query (e.g. a CAPTCHA on that page) is caught and
    logged, and the run moves on to the next query rather than dying
    entirely - so one bad (borough, band) combination doesn't sacrifice
    the rest of the run.
    """
    all_queries = [
        (location, pmin, pmax)
        for location in locations
        for (pmin, pmax) in price_bands
    ]
    completed = load_progress()
    remaining = [q for q in all_queries if query_key(*q) not in completed]
    skipped = len(all_queries) - len(remaining)
    if skipped:
        print(f"Resuming: {skipped}/{len(all_queries)} queries already checkpointed, "
              f"{len(remaining)} remaining.")
    else:
        print(f"Starting fresh: {len(all_queries)} total queries "
              f"({len(locations)} boroughs x {len(price_bands)} price bands).")

    driver = start_driver(headless=headless)
    queries_since_restart = 0

    try:
        for location, price_min, price_max in remaining:
            key = query_key(location, price_min, price_max)
            print(f"\n=== Scraping: {location} | price £{price_min or 0}-{price_max or '+'} ===")

            # Catches the case where Chrome died between queries (OS killed
            # it, tab crashed, etc.) BEFORE wasting a query on it.
            if not is_driver_alive(driver):
                print("  Browser session is dead - restarting Chrome before continuing.")
                driver = restart_driver(driver, headless=headless)
                queries_since_restart = 0

            try:
                listings = scrape_one_query(driver, location, price_min, price_max,
                                             pages_per_query, checkin, checkout)
                print(f"  -> {len(listings)} listings from this query")
                append_checkpoint(listings)
                mark_query_done(key)
            except (WebDriverException, InvalidSessionIdException) as e:
                # The crash happened DURING this query. Restart and retry it
                # once before giving up - this is the case that was silently
                # killing the rest of the run before (every remaining query
                # would hit the same dead driver and fail instantly).
                print(f"  Browser crashed mid-query on {key}: {e!r} - "
                      f"restarting Chrome and retrying this query once.")
                driver = restart_driver(driver, headless=headless)
                queries_since_restart = 0
                try:
                    listings = scrape_one_query(driver, location, price_min, price_max,
                                                 pages_per_query, checkin, checkout)
                    print(f"  -> {len(listings)} listings from this query (after retry)")
                    append_checkpoint(listings)
                    mark_query_done(key)
                except Exception as e2:
                    print(f"  ERROR on {key} after retry: {e2!r} - skipping this query, "
                          f"progress so far is safely checkpointed. Re-run to retry it.")
                    continue
            except Exception as e:
                print(f"  ERROR on {key}: {e!r} - skipping this query, "
                      f"progress so far is safely checkpointed. Re-run to retry it.")
                continue

            queries_since_restart += 1
            if queries_since_restart >= RESTART_EVERY_N_QUERIES:
                print(f"  Restarting Chrome after {RESTART_EVERY_N_QUERIES} queries "
                      f"to keep memory usage bounded for the rest of the run.")
                driver = restart_driver(driver, headless=headless)
                queries_since_restart = 0
    finally:
        try:
            driver.quit()
        except Exception:
            pass

    return load_checkpoint().to_dict("records")


def validate_output(df):
    """
    Prints a summary of the exact things that went wrong last time, so a
    stale file or silent regression gets caught here rather than being
    discovered by whoever reads the spreadsheet next.
    """
    print("\n=== Output validation summary ===")

    if "zone" in df.columns:
        blank_zones = df[df["zone"].isna()]
        if not blank_zones.empty:
            missing_locations = blank_zones["search_location"].unique().tolist()
            print(f"  WARNING: {len(blank_zones)} rows have a BLANK zone. "
                  f"Locations affected: {missing_locations}. "
                  f"Check BOROUGH_ZONE_MAP covers these before sending this file on.")
        else:
            print(f"  OK: zone is populated for all {len(df)} rows.")

    if "search_location" in df.columns:
        per_borough = df["search_location"].value_counts()
        print(f"  Listings per borough - min {per_borough.min()}, "
              f"max {per_borough.max()}, median {per_borough.median():.0f}.")
        thin = per_borough[per_borough < 50]
        if not thin.empty:
            print(f"  NOTE: {len(thin)} borough(s) under 50 listings after dedup - "
                  f"consider a narrower price band or an extra page there: "
                  f"{thin.to_dict()}")

    if "price_band_label" in df.columns:
        print("  Listings per price band:")
        print(df["price_band_label"].value_counts().to_string())

    if "rating" in df.columns:
        missing_rating = df["rating"].isna().sum()
        pct = 100 * missing_rating / len(df) if len(df) else 0
        print(f"  {missing_rating}/{len(df)} rows ({pct:.1f}%) have no rating/review_count "
              f"- check 'is_new_listing' flag; decide on imputation vs exclusion before modelling.")

    if "nights_source" in df.columns:
        counts = df["nights_source"].value_counts()
        parsed = int(counts.get("parsed_from_card", 0))
        assumed = int(counts.get("assumed_from_search_dates", 0))
        print(f"  nights_source: {parsed}/{len(df)} parsed directly from the card, "
              f"{assumed}/{len(df)} fell back to the fixed {EXPECTED_NIGHTS}-night search window.")

    if "is_hotel_brand" in df.columns:
        hotel_count = df["is_hotel_brand"].sum()
        pct = 100 * hotel_count / len(df) if len(df) else 0
        print(f"  {hotel_count}/{len(df)} rows ({pct:.1f}%) flagged as hotel/aparthotel brands "
              f"- filter these out of the peer-to-peer hedonic model, or test their effect separately.")

    if "room_id" in df.columns:
        unique_ids = df["room_id"].nunique()
        print(f"  {unique_ids}/{len(df)} rows have a unique room_id "
              f"({'OK' if unique_ids == len(df) else 'WARNING: duplicates remain - check pagination'}).")

    if "price_raw" in df.columns and "price_gbp_total" in df.columns:
        def _price_mismatch(row):
            if not row["price_raw"] or row["price_gbp_total"] is None:
                return False
            re_derived = parse_total_price(row["price_raw"])
            return re_derived is None or re_derived != row["price_gbp_total"]

        suspect_rows = df.apply(_price_mismatch, axis=1).sum()
        if suspect_rows:
            print(f"  WARNING: {suspect_rows} rows' recorded price doesn't match what "
                  f"parse_total_price(price_raw) returns now - re-check these rows individually.")
        else:
            print("  OK: every row's recorded price matches parse_total_price(price_raw).")

    print("=== End validation summary ===\n")


def save_to_excel(listings, output_file):
    df = pd.DataFrame(listings)

    # De-duplicate by room_id, NOT url or (location, band). The same
    # listing can legitimately surface in two adjacent price bands near a
    # boundary, or from a sponsored slot repeated across pages - room_id
    # is the clean numeric listing ID that catches all of these.
    if "room_id" in df.columns:
        before = len(df)
        df = df.drop_duplicates(subset="room_id")
        print(f"  Deduplication: {before} rows -> {len(df)} unique listings "
              f"({before - len(df)} duplicates removed)")

    validate_output(df)

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="listings", index=False)

        worksheet = writer.sheets["listings"]
        for i, col in enumerate(df.columns, start=1):
            max_len = max(df[col].astype(str).map(len).max() if not df.empty else 0, len(col)) + 2
            worksheet.column_dimensions[worksheet.cell(row=1, column=i).column_letter].width = min(max_len, 50)

    print(f"Saved {len(df)} unique listings to {output_file}")


if __name__ == "__main__":
    total_queries = len(LOCATIONS) * len(PRICE_BANDS)
    print(f"Starting Airbnb scrape: {len(LOCATIONS)} boroughs x {len(PRICE_BANDS)} price bands "
          f"= {total_queries} queries, {PAGES_PER_QUERY} pages each.")
    listings = scrape_airbnb(LOCATIONS, PRICE_BANDS, PAGES_PER_QUERY, CHECKIN, CHECKOUT, headless=HEADLESS)

    if listings:
        save_to_excel(listings, OUTPUT_FILE)
    else:
        print("No listings scraped. Airbnb may have changed their page structure, "
              "shown a CAPTCHA, or blocked the request. Try HEADLESS=False to watch "
              "what the browser sees.")


# ============================================================
# PAGINATION DEPTH TEST — run this BEFORE the full run
# ============================================================
def test_pagination_depth(locations, price_bands=None, max_pages=6, checkin=None, checkout=None, headless=True):
    """
    Scrapes a SMALL set of (borough, price band) queries up to max_pages,
    and reports how many NEW unique room_ids each page adds. Use this to
    confirm 2 pages/query is still the right depth (or find a new one)
    before committing to the full 33-borough x price-band run.

    Example - test 2 boroughs across all price bands:
        test_pagination_depth(
            ["Camden, London, United Kingdom", "Croydon, London, United Kingdom"],
            price_bands=PRICE_BANDS, max_pages=4, checkin=CHECKIN, checkout=CHECKOUT
        )
    """
    if price_bands is None:
        price_bands = [(None, None)]

    driver = start_driver(headless=headless)
    report = {}

    try:
        for location in locations:
            for price_min, price_max in price_bands:
                label = f"{location} | £{price_min or 0}-{price_max or '+'}"
                print(f"\n=== Testing pagination depth: {label} ===")
                listings = scrape_one_query(driver, location, price_min, price_max,
                                             max_pages, checkin, checkout)
                df = pd.DataFrame(listings)
                per_page = {}
                seen_ids = set()
                for page_num in sorted(df["page_scraped"].unique()) if not df.empty else []:
                    page_ids = set(df.loc[df["page_scraped"] == page_num, "room_id"].dropna())
                    new_ids = page_ids - seen_ids
                    per_page[int(page_num)] = len(new_ids)
                    seen_ids |= page_ids
                report[label] = per_page
                print(f"  New unique listings per page: {per_page}")
    finally:
        driver.quit()

    print("\n=== Pagination depth summary ===")
    for label, per_page in report.items():
        print(f"{label}: {per_page}")
    print("Look for the page number where new-unique-listings drops to 0 (or near it) "
          "- that's roughly where this query's results stop returning fresh listings. "
          "Set PAGES_PER_QUERY just below that point.")
    return report


Starting Airbnb scrape: 33 boroughs x 6 price bands = 198 queries, 2 pages each.
Starting fresh: 198 total queries (33 boroughs x 6 price bands).

=== Scraping: Barking and Dagenham, London, United Kingdom | price £0-75 ===
  [Page 1/2] Loading: https://www.airbnb.co.uk/s/Barking-and-Dagenham--London--United-Kingdom/homes?checkin=2026-09-01&checkout=2026-09-05&price_max=75
    Found 18 listing cards
  [Page 2/2] PAGINATION: 'Next' button found and clicked.
    PAGINATION: previous page's cards went stale - re-render confirmed.
    PAGINATION: no new cards appeared after clicking 'Next' - stopping this query.
  -> 18 listings from this query

=== Scraping: Barking and Dagenham, London, United Kingdom | price £75-120 ===
  [Page 1/2] Loading: https://www.airbnb.co.uk/s/Barking-and-Dagenham--London--United-Kingdom/homes?checkin=2026-09-01&checkout=2026-09-05&price_min=75&price_max=120
    Found 24 listing cards
  [Page 2/2] PAGINATION: 'Next' button found and clicked.
    PAGINATION WARNI